In [6]:
import src.DetectingRegionInfo as DetectingRegionInfo
import src.DopplerInfo as DopplerInfo

import src.LinesGenerator as LinesGenerator
import src.WAndDopplerGenerator as WAndDopplerGenerator
import src.FeaturesAndLabelsGenerator as FeaturesAndLabelsGenerator

import src.Config as Config
from matplotlib.patches import Polygon
import matplotlib.pyplot as plt

# create info structs from config
detecting_region_info = DetectingRegionInfo.DetectingRegionInfo(
    Config.transmittor_position, Config.receiver_position_1, Config.receiver_position_2, Config.receiver_position_3)
doppler_info = DopplerInfo.DopplerInfo(Config.c, Config.fc, Config.v)

In [7]:
def visualize_lines(detecting_region_info, lines):
    _, ax = plt.subplots()
    ax.set_xlim(0, 300)
    ax.set_ylim(0, 150)
    ax.add_patch(Polygon([detecting_region_info.v1,
                 detecting_region_info.v2, detecting_region_info.v4, detecting_region_info.v3], fill=False))

    # Plot the lines
    for i in range(len(lines)):
        line = lines[i]
        ax.plot([point[0] for point in line], [point[1]
                for point in line], "r")

    plt.show()


In [ ]:
# long testing features and labels

[lines_a, lines_b] = LinesGenerator.generateLines(
    detecting_region_info=detecting_region_info,
    a_b_distance=Config.a_b_distance,
    num_of_lines_to_generate=1,
    step_count_per_line=Config.test_set_long_num + (Config.step_count_per_line - 1),
    length_per_step=Config.length_per_step,
    angle_change_limit_per_step=Config.angle_change_limit_per_step
)

[lines_a, lines_b] = LinesGenerator.reparseSingleLineAsLines(
    line_a=lines_a[0],
    line_b=lines_b[0],
    num_of_lines_to_generate=Config.test_set_long_num,
    step_count_per_line=Config.step_count_per_line
)

[w, doppler] = WAndDopplerGenerator.generateWAndDoppler(
    detecting_region_info=detecting_region_info,
    doppler_info=doppler_info,
    lines_a=lines_a,
    lines_b=lines_b
)

[test_long_features, test_long_coor_labels] = FeaturesAndLabelsGenerator.generateFeaturesAndLabels(
    detecting_region_info=detecting_region_info,
    lines_a=lines_a,
    w=w,
    doppler=doppler,
    num_of_lines_to_generate=Config.test_set_long_num,
    step_count_per_line=Config.step_count_per_line
)
test_long_distances_labels = FeaturesAndLabelsGenerator.calculate_distances(test_long_coor_labels,detecting_region_info)

visualize_lines(detecting_region_info, lines_a)

In [9]:
import numpy as np

# the label is [the distance to detecting_region_info.transimssion_position from the point, the angle between x axis and the line from the point to detecting_region_info.transimssion_position], i want to convert to the coordinates of the point


def labels_to_coords(detecting_region_info, labels):
    ref = detecting_region_info.transmittor_position

    coords = []

    for label in labels:
        distance = label[0]
        angle = label[1]

        x = ref[0] + distance * np.cos(angle)
        y = ref[1] + distance * np.sin(angle)

        coords.append([x, y])

    return np.array(coords)


def plot_result(detecting_region_info, loss_to_show, trues, predicteds, folder_name, plot_name):
    plt.figure(figsize=(10, 6))
    plt.scatter(
        trues[:, 0],
        trues[:, 1],
        label="True",
        marker="o",
        s=30,
        alpha=0.7,
    )
    plt.scatter(
        predicteds[:, 0],
        predicteds[:, 1],
        label="Predicted",
        marker="x",
        s=30,
        alpha=0.7,
    )
    plt.xlabel("X-coordinate")
    plt.ylabel("Y-coordinate")
    plt.legend()
    plt.title("True vs. Predicted")
    plt.grid(True)
    
    plt.gca().add_patch(Polygon([detecting_region_info.v1, detecting_region_info.v2, detecting_region_info.v4, detecting_region_info.v3], fill=False))

    plt.text(
        0,
        0,
        f"Loss: {loss_to_show}",
        ha="left",
        va="bottom",
        transform=plt.gca().transAxes,
    )
    test_result_name = folder_name + '/' + plot_name
    plt.savefig(test_result_name)


In [ ]:
# 定义更深的神经网络模型
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader

import src.Config as cf
import src.TrajectoryDataset as TrajectoryDataset
import src.UavModel as UavModel
from src.kan.LBFGS import *

import os

gpu_cuda:bool = torch.cuda.is_available()
print(f'Cuda with GPU support:{gpu_cuda}')


# 创建训练和测试数据集
test_long_dataset = TrajectoryDataset.TrajectoryDataset(test_long_features, test_long_coor_labels)

# 创建 DataLoader
test_long_loader = DataLoader(test_long_dataset, batch_size=cf.test_set_long_num, shuffle=False) # MARK: i changed batch size to 1

# 使用GPU加速
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 定义模型
using_model = UavModel.UavModel()
model = using_model.to(device)

# 加载现有模型的逻辑
pretrained_model_path = "./keymodel/model.pth"  # 指定预训练模型路径
assert os.path.exists(pretrained_model_path)
print(f"Loading existing model from: {pretrained_model_path}")
model.load_state_dict(torch.load(pretrained_model_path, map_location=device))

# 打印模型参数信息
for param in model.parameters():
   print(type(param), param.size())

# 损失函数和优化器
if cf.optimizer == "Adam":
   optimizer = optim.Adam(model.parameters(), lr=cf.learning_rate)
elif cf.optimizer == "LBFGS":
   optimizer = LBFGS(
      model.parameters(),
      lr=cf.learning_rate,
      history_size=10,
      line_search_fn="strong_wolfe",
      tolerance_grad=1e-32,
      tolerance_change=1e-32,
      tolerance_ys=1e-32,
   )
criterion = nn.MSELoss()

# 训练逻辑保持不变
loss_values = []  # 存储损失值以供绘图
global total_loss

def closure():
    global total_loss
    optimizer.zero_grad()
    outputs = model(features)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    total_loss += loss.item()
    return loss

model.eval()
total_test_long_loss = 0
predicted_labels = []
true_labels = []
with torch.no_grad():
   for features, labels in test_long_loader:
      features, labels = features.to(device), labels.to(device)
      outputs = model(features)
      test_long_loss = criterion(outputs, labels)
      total_test_long_loss += test_long_loss.item()
      predicted_labels.append(outputs.cpu().numpy())
      true_labels.append(labels.cpu().numpy())


# Combine all batch predictions and ground-truth labels.
predicted_labels = np.concatenate(predicted_labels, axis=0)
true_labels = np.concatenate(true_labels, axis=0)

long_loss = total_test_long_loss / len(test_long_loader)

# If you're dealing with coordinates directly:
true_xy = true_labels
predicted_xy = predicted_labels

# Or if you're dealing with distances/angles and need to convert, do:
# true_xy = FeaturesAndLabelsGenerator.estimate_positions_optimized(true_labels, detecting_region_info)
# predicted_xy = FeaturesAndLabelsGenerator.estimate_positions_optimized(predicted_labels, detecting_region_info)

################################################################
# Single plot of all points (just like before)
################################################################
plot_result(
    detecting_region_info=detecting_region_info,
    loss_to_show=long_loss,
    trues=true_xy,
    predicteds=predicted_xy,
    folder_name="./tmp",
    plot_name=f"der.png"
)
print(f"TestLong Loss: {long_loss}")

################################################################
# Create multiple plots, each with a growing subset of points
################################################################

# You can decide how many frames/plots you want to make. 
# For example, you might want exactly len(true_xy) plots.
# Or if you want strictly Config.test_set_long_num plots, replace 
# len(true_xy) with Config.test_set_long_num (if that matches dimension).
num_plots = min(len(true_xy), Config.test_set_long_num)

for i in range(1, num_plots + 1):
   # Pull the first i points from the GT and from the predictions
   partial_true_xy = true_xy[:i, :]
   partial_predicted_xy = predicted_xy[:i, :]

   # Optionally compute a partial MSE for the first i points
   partial_loss = np.mean((partial_true_xy - partial_predicted_xy)**2)

   # Generate a distinct filename for each figure
   partial_plot_name = f"der_{i}.png"

   # Plot them
   plot_result(
      detecting_region_info=detecting_region_info,
      loss_to_show=partial_loss,
      trues=partial_true_xy,
      predicteds=partial_predicted_xy,
      folder_name="./tmp",
      plot_name=partial_plot_name
   )

   print(f"Saved partial plot {i}/{num_plots} with partial loss {partial_loss}")
